In [46]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path

In [47]:
RESULTS_DIR = "../results"
IMAGES_DIR = "./images"
Path(IMAGES_DIR).mkdir(parents=True, exist_ok=True)

os.listdir(RESULTS_DIR)

['phi4_latest_sync_chain_results_validation.csv',
 'gemma3_latest_sync_chain_results_validation.csv',
 'gemma3_latest_async_chain_results_validation.csv',
 'llama3_2_latest_sync_chain_results_validation.csv',
 'phi4_latest_async_chain_results_validation.csv',
 'mistral_latest_async_chain_results_validation.csv',
 'llama3_2_latest_async_chain_results_validation.csv',
 'mistral_latest_sync_chain_results_validation.csv']

In [48]:
def load_benchmark_df(results_dir: str) -> pd.DataFrame:
    benchmark_df = pd.DataFrame()
    for filename in os.listdir(results_dir):
        df = pd.read_csv(os.path.join(results_dir, filename))
        model_name = "_".join(filename.split("/")[-1].split("_")[0:2])
        df["ModelName"] = model_name
        is_async = "async" in filename
        df["Algorithm"] = "Async" if is_async else "Sync"
        benchmark_df = pd.concat([benchmark_df, df])
    return benchmark_df

benchmark_df = load_benchmark_df(RESULTS_DIR)
benchmark_df.head()

,RowID,ErrorOccurred,ExecutionOutput,HasTimedOut,ExecutionError,ExecutionTime,ExpectedOutput,ActualOutput,CorrectOutput,FirstOutput,ModelName,Algorithm
0,511,False,22\n,False,NaN,6.839299,15,22,False,6.839018,phi4_latest,Sync
1,511,False,3\n,False,NaN,9.471492,2,3,False,9.471236,phi4_latest,Sync
2,512,False,"{6: 2, 7: 2, 8: 1, 9: 1, 10: 2}\n",False,NaN,6.843503,"{6: 2, 7: 2, 8: 1, 9: 1, 10: 2}","{6: 2, 7: 2, 8: 1, 9: 1, 10: 2}",True,6.843277,phi4_latest,Sync
3,512,False,"{7: 2, 8: 2, 9: 1, 10: 1, 11: 2}\n",False,NaN,6.491076,"{7: 2, 8: 2, 9: 1, 10: 1, 11: 2}","{7: 2, 8: 2, 9: 1, 10: 1, 11: 2}",True,6.490887,phi4_latest,Sync
4,513,False,"[7, 'PF', 8, 'PF', 9, 'PF', 10, 'PF']\n",False,NaN,4.194599,"[7, 'PF', 8, 'PF', 9, 'PF', 10, 'PF']","[7, 'PF', 8, 'PF', 9, 'PF', 10, 'PF']",True,4.194437,phi4_latest,Sync


In [49]:
def display_execution_time_describe(df: pd.DataFrame) -> None:
    print(df.groupby(["ModelName", "Algorithm"])["ExecutionTime"].describe())

display_execution_time_describe(benchmark_df)

                          count      mean       std       min       25%  \
ModelName      Algorithm                                                  
gemma3_latest  Async      113.0  3.118605  6.518393  0.605181  1.155431   
               Sync       113.0  3.101047  6.468311  0.604042  1.150314   
llama3_2       Async      113.0  0.544600  0.520192  0.207981  0.357701   
               Sync       113.0  5.736692  5.620294  0.710194  1.530711   
mistral_latest Async      113.0  2.477018  1.014996  1.140326  1.846438   
               Sync       113.0  2.557761  0.893444  1.106978  1.936649   
phi4_latest    Async      113.0  5.757169  2.379056  3.388267  4.454360   
               Sync       113.0  6.198497  3.157714  3.583039  4.591494   

                               50%       75%        max  
ModelName      Algorithm                                 
gemma3_latest  Async      1.356923  1.852362  31.368442  
               Sync       1.337192  1.854031  30.013496  
llama3_2       As

In [50]:
def save_summary_tables(df: pd.DataFrame, tables_dir: str) -> None:
    Path(tables_dir).mkdir(parents=True, exist_ok=True)
    grouped = df.groupby(["ModelName", "Algorithm", "ErrorOccurred"]).agg(
        Count=("ExecutionTime", "count"),
        Avg_ExecTime=("ExecutionTime", "mean"),
        Median_ExecTime=("ExecutionTime", "median"),
        Avg_FirstOutput=("FirstOutput", "mean"),
        Median_FirstOutput=("FirstOutput", "median")
    ).reset_index()
    grouped.to_csv(os.path.join(tables_dir, "summary_by_error.csv"), index=False)

save_summary_tables(benchmark_df, "./tables")

def display_summary_tables(df: pd.DataFrame) -> None:
    grouped = df.groupby(["ModelName", "Algorithm", "ErrorOccurred"]).agg(
        Count=("ExecutionTime", "count"),
        Avg_ExecTime=("ExecutionTime", "mean"),
        Median_ExecTime=("ExecutionTime", "median"),
        Avg_FirstOutput=("FirstOutput", "mean"),
        Median_FirstOutput=("FirstOutput", "median")
    ).reset_index()
    non_error_df = df[df["ErrorOccurred"] == False]
    grouped_correct = non_error_df.groupby(["ModelName", "Algorithm", "CorrectOutput"]).agg(
        Count=("ExecutionTime", "count"),
        Avg_ExecTime=("ExecutionTime", "mean"),
        Median_ExecTime=("ExecutionTime", "median"),
        Avg_FirstOutput=("FirstOutput", "mean"),
        Median_FirstOutput=("FirstOutput", "median")
    ).reset_index()
    error_summary = df.groupby(["ModelName", "Algorithm"]).agg(
        ErrorCount=("ErrorOccurred", lambda x: (x == True).sum())
    ).reset_index()
    final = pd.merge(grouped, error_summary, on=["ModelName", "Algorithm"], how="left")
    print("Summary by ModelName, Algorithm, and ErrorOccurred:")
    display(final)
    print("\nSummary by ModelName, Algorithm, and CorrectOutput (non-error cases only):")
    display(grouped_correct)

display_summary_tables(benchmark_df)

Summary by ModelName, Algorithm, and ErrorOccurred:


,ModelName,Algorithm,ErrorOccurred,Count,Avg_ExecTime,Median_ExecTime,Avg_FirstOutput,Median_FirstOutput,ErrorCount
0,gemma3_latest,Async,False,111,3.154407,1.356923,2.552142,1.077187,2
1,gemma3_latest,Async,True,2,1.131544,1.131544,0.947336,0.947336,2
2,gemma3_latest,Sync,False,111,3.135730,1.337192,3.135597,1.337061,2
3,gemma3_latest,Sync,True,2,1.176103,1.176103,1.176103,1.176103,2
4,llama3_2,Async,False,107,0.492344,0.476529,0.206955,0.206368,6
5,llama3_2,Async,True,6,1.476487,0.708811,0.188967,0.175218,6
6,llama3_2,Sync,False,108,5.375909,3.006472,5.363604,3.003628,5
7,llama3_2,Sync,True,5,13.529605,7.579711,13.529605,7.579711,5
8,mistral_latest,Async,False,104,2.490192,2.222190,0.850568,0.690420,9
9,mistral_latest,Async,True,9,2.324779,2.261999,1.580108,1.309868,9



Summary by ModelName, Algorithm, and CorrectOutput (non-error cases only):


,ModelName,Algorithm,CorrectOutput,Count,Avg_ExecTime,Median_ExecTime,Avg_FirstOutput,Median_FirstOutput
0,gemma3_latest,Async,False,51,5.129897,1.522269,4.194038,1.214171
1,gemma3_latest,Async,True,60,1.475241,1.277992,1.156531,0.954665
2,gemma3_latest,Sync,False,51,5.097287,1.512076,5.097157,1.511915
3,gemma3_latest,Sync,True,60,1.468407,1.276511,1.468271,1.276381
4,llama3_2,Async,False,46,0.536073,0.532943,0.213189,0.187951
5,llama3_2,Async,True,61,0.459369,0.457538,0.202255,0.207933
6,llama3_2,Sync,False,46,6.342646,3.822851,6.333716,3.819914
7,llama3_2,Sync,True,62,4.658652,2.624776,4.643843,2.624412
8,mistral_latest,Async,False,76,2.632836,2.358467,0.803440,0.608976
9,mistral_latest,Async,True,28,2.103016,1.974233,0.978486,0.893184


In [51]:
def plot_execution_time_catplot(df: pd.DataFrame, image_path: str) -> None:
    sns.set_style(style="whitegrid")
    g_exec = sns.catplot(
        data=df, 
        x="Algorithm", 
        y="ExecutionTime", 
        hue="ErrorOccurred", 
        col="ModelName", 
        kind="box",
        height=4, 
        aspect=0.8,
        palette="Set2",
        sharey=False
    )
    g_exec.set_titles("Model: {col_name}")
    g_exec.figure.suptitle("Execution Time by Algorithm and Error Occurrence", y=1.05)
    g_exec.set_axis_labels("Algorithm", "Execution Time (s)")
    plt.tight_layout()
    g_exec.savefig(image_path)
    plt.close()

def plot_first_output_catplot(df: pd.DataFrame, image_path: str) -> None:
    sns.set_style(style="whitegrid")
    g_first = sns.catplot(
        data=df, 
        x="Algorithm", 
        y="FirstOutput", 
        hue="ErrorOccurred", 
        col="ModelName", 
        kind="box",
        height=4, 
        aspect=0.8,
        palette="Set2",
        sharey=False
    )
    g_first.set_titles("Model: {col_name}")
    g_first.figure.suptitle("First Output Time by Algorithm and Error Occurrence", y=1.05)
    g_first.set_axis_labels("Algorithm", "First Output Time (s)")
    plt.tight_layout()
    g_first.savefig(image_path)
    plt.close()

plot_execution_time_catplot(benchmark_df, os.path.join(IMAGES_DIR, "execution_time_catplot.png"))
plot_first_output_catplot(benchmark_df, os.path.join(IMAGES_DIR, "first_output_catplot.png"))

In [52]:
def plot_execution_time_hist(df: pd.DataFrame, image_path: str) -> None:
    plt.figure(figsize=(10, 6))
    async_df = df[df["Algorithm"] == "Async"]
    sync_df = df[df["Algorithm"] == "Sync"]
    sns.histplot(x=async_df["ExecutionTime"], bins=20, kde=True, label="Async")
    sns.histplot(x=sync_df["ExecutionTime"], bins=20, kde=True, label="Sync")
    plt.title("Distribution of Execution Times")
    plt.xlabel("Execution Time (seconds)")
    plt.ylabel("Frequency")
    plt.legend()
    plt.tight_layout()
    plt.savefig(image_path)
    plt.close()

plot_execution_time_hist(benchmark_df, os.path.join(IMAGES_DIR, "execution_time_hist.png"))

In [53]:
def plot_first_output_hist(df: pd.DataFrame, image_path: str) -> None:
    plt.figure(figsize=(10, 6))
    async_df = df[df["Algorithm"] == "Async"]
    sync_df = df[df["Algorithm"] == "Sync"]
    sns.histplot(x=async_df["FirstOutput"], bins=20, kde=True, label="Async")
    sns.histplot(x=sync_df["FirstOutput"], bins=20, kde=True, label="Sync")
    plt.title("Distribution of First Output Times")
    plt.xlabel("First Output Time (seconds)")
    plt.ylabel("Frequency")
    plt.legend()
    plt.tight_layout()
    plt.savefig(image_path)
    plt.close()

plot_first_output_hist(benchmark_df, os.path.join(IMAGES_DIR, "first_output_hist.png"))

In [54]:
def plot_execution_time_boxplot(df: pd.DataFrame, image_path: str) -> None:
    fig, ax = plt.subplots(figsize=(10, 6))
    df.boxplot(column="ExecutionTime", by=["ErrorOccurred", "Algorithm"], ax=ax)
    plt.title("Execution Time by Error Occurrence and Algorithm")
    plt.suptitle("")
    plt.xlabel("Error Occurred and Algorithm")
    plt.ylabel("Execution Time (seconds)")
    plt.tight_layout()
    plt.savefig(image_path)
    plt.close()

plot_execution_time_boxplot(benchmark_df, os.path.join(IMAGES_DIR, "execution_time_boxplot.png"))